# 🎛️ YuE2 Studio — AI Music Producer (Colab T4 Edition)

**2-minute song in ~90 seconds** with GGUF Q4 quantized inference on a free Tesla T4 GPU.

* ⚡ **Pre-compiled audio.cpp CUDA engine** (instant 2s setup, skips 15min build)
* 🤖 **AI Music Producer Chat** (conversational track iteration & live audio playback)
* 🌐 **Cloudflare Tunnel** (zero-config public HTTPS link for mobile & desktop)

> ⚠️ **STEP 0 (Enable GPU):** Click **Runtime** ➔ **Change runtime type** ➔ **T4 GPU** ➔ **Save**.

In [ ]:
#@title 🚀 1. Install YuE2 Studio + GGUF Engine + Models (1-Click Setup) ⚙️
#@markdown GitHub Token (Optional — only required if repo is currently set to Private)
github_token = "" #@param {type:"string"}

import os, sys, time, subprocess, shutil, json
import urllib.request

# ── Resolve Token (Optional fallback to Colab Secret) ─────────
PAT = github_token.strip()
if not PAT:
    try:
        from google.colab import userdata
        PAT = userdata.get('GITHUB_PAT') or ''
    except Exception:
        pass

REPO = 'BartSkol/Yue2_Studio'
RELEASE_TAG = 'v0.1.0-colab-gpu'
ASSET_NAME = 'audiocpp_cli-linux-cuda.tar.gz'

# ── CUDA env ────────────────────────────────────────────────
if os.path.exists('/usr/local/cuda'):
    os.environ['CUDA_HOME'] = '/usr/local/cuda'
    os.environ['PATH'] = f"/usr/local/cuda/bin:{os.environ.get('PATH', '')}"
    os.environ['LD_LIBRARY_PATH'] = f"/usr/local/cuda/lib64:{os.environ.get('LD_LIBRARY_PATH', '')}"
os.environ['TRANSFORMERS_NO_TORCHVISION'] = '1'

t0 = time.time()

# ── [1/4] Install Python packages (background) ──────────────
print('📦 [1/4] Installing Python dependencies (background)...')
pip_proc = subprocess.Popen(
    'pip uninstall -y -q torchvision 2>/dev/null; '
    'pip install -q --upgrade pip setuptools wheel && '
    'pip install -q --upgrade '
    '"transformers[torch]>=4.40.0" accelerate safetensors einops '
    '"soundfile>=0.12.1" "numpy>=1.24.0" "tiktoken>=0.7.0" '
    '"huggingface_hub>=0.26.0" "diffusers>=0.30.0" "scipy>=1.11.0" cmake ninja peft pillow',
    shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE
)

# ── [2/4] Download pre-compiled audiocpp_cli ────────────────
print('⚡ [2/4] Downloading pre-compiled audiocpp_cli CUDA engine...')
bin_downloaded = False
if not os.path.exists('/content/YuE/audiocpp_cli'):
    tarball = f'/tmp/{ASSET_NAME}'
    # Try public release download first
    try:
        subprocess.run([
            'wget', '-q', '-nc',
            f'https://github.com/{REPO}/releases/download/{RELEASE_TAG}/{ASSET_NAME}',
            '-O', tarball
        ], check=True)
        if os.path.isfile(tarball) and os.path.getsize(tarball) > 1_000_000:
            bin_downloaded = True
            print(f'   ✅ Downloaded pre-compiled binary ({os.path.getsize(tarball)/1e6:.0f} MB)')
    except Exception:
        pass

    # Fallback to API with token if repo/release is private
    if not bin_downloaded and PAT:
        try:
            api_url = f'https://api.github.com/repos/{REPO}/releases/tags/{RELEASE_TAG}'
            req = urllib.request.Request(api_url, headers={'Authorization': f'token {PAT}', 'Accept': 'application/vnd.github.v3+json'})
            with urllib.request.urlopen(req, timeout=30) as resp:
                release_data = json.loads(resp.read())
            asset = next((a for a in release_data.get('assets', []) if a['name'] == ASSET_NAME), None)
            if asset:
                dl_req = urllib.request.Request(asset['url'], headers={'Authorization': f'token {PAT}', 'Accept': 'application/octet-stream'})
                with urllib.request.urlopen(dl_req, timeout=120) as r, open(tarball, 'wb') as f:
                    shutil.copyfileobj(r, f)
                if os.path.getsize(tarball) > 1_000_000:
                    bin_downloaded = True
                    print(f'   ✅ Downloaded via API ({os.path.getsize(tarball)/1e6:.0f} MB)')
        except Exception as e:
            print(f'   ⚠️ API Release download note: {e}')

# ── [3/4] Clone YuE2 + Studio repositories ──────────────────
print('📦 [3/4] Setting up YuE2 Studio repository...')
if not os.path.exists('/content/YuE'):
    !git clone -q https://github.com/multimodal-art-projection/YuE.git /content/YuE
    %cd /content/YuE
    !git checkout -q 92a73cc7652fcc1f937855e4b765e0a0edd7ff2e

if not os.path.exists('/content/YuE/Yue2_Studio'):
    clone_url = f'https://{PAT}@github.com/{REPO}.git' if PAT else f'https://github.com/{REPO}.git'
    !git clone -q {clone_url} /content/YuE/Yue2_Studio

%cd /content/YuE
!python /content/YuE/Yue2_Studio/install_studio.py --target /content/YuE
!cp /content/YuE/Yue2_Studio/launch_colab.py /content/YuE/launch_colab.py

# Extract pre-compiled binary
tarball = f'/tmp/{ASSET_NAME}'
if bin_downloaded and os.path.isfile(tarball) and not os.path.isfile('/content/YuE/audiocpp_cli'):
    !tar -xzf {tarball} -C /content/YuE/
    !chmod +x /content/YuE/audiocpp_cli
    !cp /content/YuE/audiocpp_cli /usr/local/bin/audiocpp_cli
    print('   ✅ audiocpp_cli installed')

# T4 SDPA fallback patch
cuda_graph_py = '/content/YuE/src/yue2/cuda_graph.py'
if os.path.exists(cuda_graph_py):
    with open(cuda_graph_py, 'r') as f:
        c = f.read()
    c = c.replace('flash = fused and torch.backends.cuda.is_flash_attention_available()', 'cap = torch.cuda.get_device_capability(self.device) if self.device.type == "cuda" else (0, 0)\n        flash = fused and cap[0] >= 8 and torch.backends.cuda.is_flash_attention_available()')
    c = c.replace('attention_backend = "flash" if flash else "cudnn" if fused and torch.backends.cudnn.is_available() else "sdpa"', 'attention_backend = "flash" if flash else "sdpa"')
    with open(cuda_graph_py, 'w') as f:
        f.write(c)

# Wait for pip to finish
pip_proc.wait()
!pip install -q -e /content/YuE 2>/dev/null

for d in ('models/YuE2-3B', 'models/YuE2-Vae', 'models/Yue2-3B-GGUF', 'models/loras', 'runs/studio'):
    os.makedirs(f'/content/YuE/{d}', exist_ok=True)

# ── [4/4] Download GGUF Q4 bundle + LoRA ────────────────────
print('📥 [4/4] Downloading GGUF Q4 model weights & VAE...')
!huggingface-cli download audio-cpp/Yue2-3B-GGUF yue2-3b-q4_0.gguf --local-dir /content/YuE/models/Yue2-3B-GGUF --quiet
!huggingface-cli download audio-cpp/Yue2-3B-GGUF yue2-vae-f16.gguf --local-dir /content/YuE/models/Yue2-3B-GGUF --quiet
!huggingface-cli download audio-cpp/Yue2-3B-GGUF --include "sidecars/*" --local-dir /content/YuE/models/Yue2-3B-GGUF --quiet
!huggingface-cli download Mothersuperior/YuE2-instrumental-cot-full-loras --local-dir /content/YuE/models/loras/YuE2-instrumental-cot --quiet

# ── Summary ─────────────────────────────────────────────────
elapsed = time.time() - t0
import torch
print(f'\n{"="*55}')
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB VRAM)')
print(f'⚡ Fast presets: audio.cpp + CUDA + Q4 + 16 Steps + CFG 1.0 (~90s / 2-min song)')
print(f'✅ Full environment installed and ready in {elapsed:.0f}s!')
print(f'{"="*55}')

--- 
## 🤖 2. (Optional) Start Local AI Assistant (Qwen 2.5 3B with GPU)

In [ ]:
#@title 🤖 2. Start Local AI Assistant (Qwen 2.5 3B on GPU) ✍️
#@markdown Enables zero-cost AI Music Producer Chat & lyrics writing inside Studio WebUI
model_qwen = "qwen2.5:3b" #@param ["qwen2.5:3b", "qwen2.5:1.5b", "qwen2.5:0.5b", "llama3.2:3b"]

import os, subprocess, time, shutil as _sh

print('📦 Installing GPU tools (pciutils, lshw, zstd)...')
!apt-get update -qq && apt-get install -y -qq pciutils lshw zstd > /dev/null 2>&1

print('🤖 Installing Ollama engine with GPU acceleration...')
!curl -fsSL https://ollama.com/install.sh | sh

ollama_bin = _sh.which('ollama') or '/usr/local/bin/ollama'
if os.path.isfile(ollama_bin):
    # Start Ollama service
    subprocess.Popen([ollama_bin, 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(4)
    print(f'📥 Pulling {model_qwen} model into GPU VRAM (~1.9 GB)...')
    !{ollama_bin} pull {model_qwen}
    print(f'\n✅ AI Producer & Songwriter ({model_qwen}) is LIVE on GPU!')
else:
    print('❌ Ollama installation failed.')

--- 
## 🎛️ 3. Launch YuE2 Studio

In [ ]:
#@title 🎛️ 3. Start YuE2 Studio (Public Cloudflare HTTPS Link) 🚀
%cd /content/YuE

# Kill any previous hanging instance
!fuser -k 7862/tcp > /dev/null 2>&1 || true
!pkill -9 -f launch_colab.py > /dev/null 2>&1 || true

# Start Studio server
!python -u launch_colab.py